# Train vs Synthetic vs Production Distribution Comparison

This notebook compares simple IMU sensor distributions before moving to Stage 4 inference.

Datasets:
- **train_all_users**: `data/processed/fused_all_users_data_labeled_50Hz.csv`
- **synthetic_codex**: raw-like labeled synthetic all-users data
- **synthetic_other**: another labeled synthetic candidate
- **synthetic_production**: unlabeled production-shaped stress data
- **production_all**: all processed production `fused_20*_50Hz.csv` sessions

Sensor space used here:
- accelerometer in `m/s^2`
- gyroscope in `deg/s`
- canonical columns: `Ax, Ay, Az, Gx, Gy, Gz`
- no z-score, no gravity removal

In [ ]:
from pathlib import Path
import json
import math
import re
import zlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

OUT_DIR = ROOT / "reports" / "data_distribution_comparison"
CHART_DIR = OUT_DIR / "charts"
OUT_DIR.mkdir(parents=True, exist_ok=True)
CHART_DIR.mkdir(parents=True, exist_ok=True)

SENSOR_COLS = ["Ax", "Ay", "Az", "Gx", "Gy", "Gz"]
ACCEL_COLS = ["Ax", "Ay", "Az"]
GYRO_COLS = ["Gx", "Gy", "Gz"]
FEATURE_COLS = SENSOR_COLS + ["A_mag", "G_mag"]

MILLI_G_TO_MPS2 = 9.80665 / 1000.0
CHUNK_SIZE = 200_000
MAX_SAMPLE_PER_FILE = 12_000
MAX_SAMPLE_PER_DATASET = 260_000
RNG_SEED = 20260502

COLUMN_ALIASES = {
    "Ax": ["Ax", "Ax_w", "accel_x", "x_accel", "acceleration_x"],
    "Ay": ["Ay", "Ay_w", "accel_y", "y_accel", "acceleration_y"],
    "Az": ["Az", "Az_w", "accel_z", "z_accel", "acceleration_z"],
    "Gx": ["Gx", "Gx_w", "gyro_x", "x_gyro", "gyroscope_x"],
    "Gy": ["Gy", "Gy_w", "gyro_y", "y_gyro", "gyroscope_y"],
    "Gz": ["Gz", "Gz_w", "gyro_z", "z_gyro", "gyroscope_z"],
}

print("Repo root:", ROOT)
print("Output:", OUT_DIR)

In [ ]:
def discover_production_files():
    files = []
    for path in (ROOT / "data" / "processed").glob("fused_20*.csv"):
        name = path.name.lower()
        if any(skip in name for skip in ["synthetic", "all_users", "train", "test", "val"]):
            continue
        files.append(path)
    return sorted(files)

def discover_synthetic_production_files():
    return sorted((ROOT / "data" / "synthetic_production").glob("synthetic_production_*.csv"))

datasets = {
    "train_all_users": [ROOT / "data" / "processed" / "fused_all_users_data_labeled_50Hz.csv"],
    "synthetic_codex": [ROOT / "data" / "processed" / "fused_synthetic_codex_input_50Hz.csv"],
    "synthetic_other": [ROOT / "data" / "processed" / "fused_synthetic_labeled_50Hz.csv"],
    "synthetic_production": discover_synthetic_production_files(),
    "production_all": discover_production_files(),
}

for name, files in datasets.items():
    print(f"{name:22s} files={len(files):2d}")
    for p in files[:5]:
        print("  ", p.relative_to(ROOT))
    if len(files) > 5:
        print("   ...")

In [ ]:
class RunningStats:
    def __init__(self):
        self.count = {c: 0 for c in FEATURE_COLS}
        self.missing = {c: 0 for c in FEATURE_COLS}
        self.total = {c: 0.0 for c in FEATURE_COLS}
        self.total_sq = {c: 0.0 for c in FEATURE_COLS}
        self.min_val = {c: math.inf for c in FEATURE_COLS}
        self.max_val = {c: -math.inf for c in FEATURE_COLS}

    def update(self, frame):
        for col in FEATURE_COLS:
            values = pd.to_numeric(frame[col], errors="coerce")
            self.missing[col] += int(values.isna().sum())
            valid = values.dropna().to_numpy(dtype=np.float64)
            if len(valid) == 0:
                continue
            self.count[col] += len(valid)
            self.total[col] += float(valid.sum())
            self.total_sq[col] += float(np.dot(valid, valid))
            self.min_val[col] = min(self.min_val[col], float(valid.min()))
            self.max_val[col] = max(self.max_val[col], float(valid.max()))

    def merge(self, other):
        for col in FEATURE_COLS:
            self.count[col] += other.count[col]
            self.missing[col] += other.missing[col]
            self.total[col] += other.total[col]
            self.total_sq[col] += other.total_sq[col]
            self.min_val[col] = min(self.min_val[col], other.min_val[col])
            self.max_val[col] = max(self.max_val[col], other.max_val[col])

    def rows(self, dataset, sample):
        out = []
        for col in FEATURE_COLS:
            n = self.count[col]
            mean = self.total[col] / n if n else np.nan
            var = (self.total_sq[col] / n) - mean * mean if n else np.nan
            std = math.sqrt(max(var, 0.0)) if n else np.nan
            q = sample[col].quantile([0.05, 0.25, 0.50, 0.75, 0.95]) if len(sample) else {}
            out.append({
                "dataset": dataset,
                "feature": col,
                "count": n,
                "missing": self.missing[col],
                "mean": mean,
                "std": std,
                "min": self.min_val[col] if n else np.nan,
                "p05": float(q.loc[0.05]) if len(sample) else np.nan,
                "p25": float(q.loc[0.25]) if len(sample) else np.nan,
                "median": float(q.loc[0.50]) if len(sample) else np.nan,
                "p75": float(q.loc[0.75]) if len(sample) else np.nan,
                "p95": float(q.loc[0.95]) if len(sample) else np.nan,
                "max": self.max_val[col] if n else np.nan,
            })
        return out

def canonical_column_map(path):
    header = pd.read_csv(path, nrows=0)
    lower_to_original = {str(c).strip().lower(): c for c in header.columns}
    exact = {str(c).strip(): c for c in header.columns}
    mapping = {}
    for canonical, candidates in COLUMN_ALIASES.items():
        found = None
        for candidate in candidates:
            if candidate in exact:
                found = exact[candidate]
                break
            if candidate.lower() in lower_to_original:
                found = lower_to_original[candidate.lower()]
                break
        if found is None:
            raise ValueError(f"{path} missing {canonical}")
        mapping[found] = canonical
    return mapping

def needs_accel_conversion(path, usecols, rename_map):
    preview = pd.read_csv(path, usecols=usecols, nrows=5000).rename(columns=rename_map)
    preview = preview[SENSOR_COLS].apply(pd.to_numeric, errors="coerce")
    median_a_mag = float(np.nanmedian(np.sqrt(np.square(preview[ACCEL_COLS]).sum(axis=1))))
    return median_a_mag > 80.0

def clean_chunk(chunk, rename_map, convert_accel):
    frame = chunk.rename(columns=rename_map)[SENSOR_COLS].copy()
    frame = frame.apply(pd.to_numeric, errors="coerce")
    if convert_accel:
        frame[ACCEL_COLS] = frame[ACCEL_COLS] * MILLI_G_TO_MPS2
    frame["A_mag"] = np.sqrt(np.square(frame[ACCEL_COLS]).sum(axis=1))
    frame["G_mag"] = np.sqrt(np.square(frame[GYRO_COLS]).sum(axis=1))
    return frame

def sample_frame(frame, n, rng):
    if len(frame) <= n:
        return frame
    return frame.sample(n=n, random_state=int(rng.integers(0, 2**31 - 1)))

In [ ]:
def process_dataset(name, files, per_file=False):
    rng = np.random.default_rng(RNG_SEED + zlib.crc32(name.encode("utf-8")) % 100_000)
    aggregate = RunningStats()
    samples = []
    per_file_stats = {}
    meta = []

    for path in files:
        rename_map = canonical_column_map(path)
        usecols = list(rename_map.keys())
        convert_accel = needs_accel_conversion(path, usecols, rename_map)
        file_stats = RunningStats()
        file_samples = []

        for raw_chunk in pd.read_csv(path, usecols=usecols, chunksize=CHUNK_SIZE):
            chunk = clean_chunk(raw_chunk, rename_map, convert_accel)
            file_stats.update(chunk)
            if sum(len(s) for s in file_samples) < MAX_SAMPLE_PER_FILE:
                remaining = MAX_SAMPLE_PER_FILE - sum(len(s) for s in file_samples)
                target = min(remaining, max(500, MAX_SAMPLE_PER_FILE // 8), len(chunk))
                file_samples.append(sample_frame(chunk, target, rng))

        aggregate.merge(file_stats)
        if file_samples:
            samples.append(pd.concat(file_samples, ignore_index=True))
        if per_file:
            per_file_stats[path.stem] = file_stats
        meta.append({
            "dataset": name,
            "file": str(path.relative_to(ROOT)),
            "rows": file_stats.count["Ax"],
            "accel_converted_millig_to_mps2": convert_accel,
        })

    sample = pd.concat(samples, ignore_index=True) if samples else pd.DataFrame(columns=FEATURE_COLS)
    if len(sample) > MAX_SAMPLE_PER_DATASET:
        sample = sample_frame(sample, MAX_SAMPLE_PER_DATASET, rng).reset_index(drop=True)
    return aggregate, sample, per_file_stats, meta

all_stats_rows = []
samples = {}
file_meta = []
production_per_file = {}

for name, files in datasets.items():
    print("Processing", name, "files=", len(files))
    stats, sample, per_file_stats, meta = process_dataset(name, files, per_file=(name == "production_all"))
    samples[name] = sample
    file_meta.extend(meta)
    all_stats_rows.extend(stats.rows(name, sample))
    if name == "production_all":
        production_per_file = per_file_stats

stats_df = pd.DataFrame(all_stats_rows)
stats_df.to_csv(OUT_DIR / "distribution_stats.csv", index=False)
pd.DataFrame(file_meta).to_json(OUT_DIR / "source_files_and_unit_conversion.json", orient="records", indent=2)

stats_df[stats_df.feature.isin(["A_mag", "G_mag"])][["dataset", "feature", "mean", "std", "p05", "median", "p95"]]

In [ ]:
def wasserstein_approx(a, b, n=5000):
    a = np.sort(a[np.isfinite(a)])
    b = np.sort(b[np.isfinite(b)])
    if len(a) == 0 or len(b) == 0:
        return np.nan
    if len(a) > n:
        a = a[np.linspace(0, len(a) - 1, n).astype(int)]
    if len(b) > n:
        b = b[np.linspace(0, len(b) - 1, n).astype(int)]
    m = min(len(a), len(b))
    qa = np.interp(np.linspace(0, 1, m), np.linspace(0, 1, len(a)), a)
    qb = np.interp(np.linspace(0, 1, m), np.linspace(0, 1, len(b)), b)
    return float(np.mean(np.abs(qa - qb)))

def psi(ref, cur, bins=10):
    ref = ref[np.isfinite(ref)]
    cur = cur[np.isfinite(cur)]
    if len(ref) == 0 or len(cur) == 0:
        return np.nan
    edges = np.unique(np.quantile(ref, np.linspace(0, 1, bins + 1)))
    if len(edges) < 3:
        return 0.0
    edges[0], edges[-1] = -np.inf, np.inf
    ref_counts, _ = np.histogram(ref, bins=edges)
    cur_counts, _ = np.histogram(cur, bins=edges)
    ref_pct = np.clip(ref_counts / max(ref_counts.sum(), 1), 1e-6, None)
    cur_pct = np.clip(cur_counts / max(cur_counts.sum(), 1), 1e-6, None)
    return float(np.sum((cur_pct - ref_pct) * np.log(cur_pct / ref_pct)))

lookup = {
    (row.dataset, row.feature): row
    for row in stats_df.itertuples(index=False)
}

rows = []
baseline = "train_all_users"
for dataset in samples:
    if dataset == baseline:
        continue
    for feature in FEATURE_COLS:
        ref = lookup[(baseline, feature)]
        cur = lookup[(dataset, feature)]
        std_ratio = cur.std / ref.std if ref.std > 1e-12 else np.nan
        rows.append({
            "reference": baseline,
            "dataset": dataset,
            "feature": feature,
            "mean_delta": cur.mean - ref.mean,
            "mean_delta_in_train_std": abs(cur.mean - ref.mean) / ref.std if ref.std > 1e-12 else np.nan,
            "std_ratio": std_ratio,
            "wasserstein_approx": wasserstein_approx(samples[baseline][feature].to_numpy(float), samples[dataset][feature].to_numpy(float)),
            "psi_vs_train": psi(samples[baseline][feature].to_numpy(float), samples[dataset][feature].to_numpy(float)),
        })

shift_df = pd.DataFrame(rows)
shift_df["simple_shift_score"] = (
    shift_df["mean_delta_in_train_std"].fillna(0)
    + np.abs(np.log(shift_df["std_ratio"].replace(0, np.nan))).fillna(0)
    + shift_df["psi_vs_train"].fillna(0)
)
shift_df.to_csv(OUT_DIR / "shift_vs_train.csv", index=False)

shift_df.sort_values("simple_shift_score", ascending=False).head(15)

In [ ]:
def plot_grouped_bars(metric, features, path, title):
    pivot = stats_df[stats_df.feature.isin(features)].pivot(index="feature", columns="dataset", values=metric)
    ax = pivot.plot(kind="bar", figsize=(12, 6), width=0.82)
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel(metric)
    ax.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.savefig(path, dpi=160)
    plt.show()

def plot_hist_overlay(feature, path, title):
    plt.figure(figsize=(11, 6))
    for dataset, sample in samples.items():
        values = sample[feature].dropna()
        lo, hi = np.nanpercentile(values, [1, 99])
        plt.hist(values.clip(lo, hi), bins=80, density=True, histtype="step", linewidth=1.6, label=dataset)
    plt.title(title)
    plt.xlabel(feature)
    plt.ylabel("density")
    plt.grid(alpha=0.2)
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(path, dpi=160)
    plt.show()

def plot_shift_heatmap(path):
    pivot = shift_df.pivot(index="dataset", columns="feature", values="psi_vs_train")
    fig, ax = plt.subplots(figsize=(11, 4.8))
    im = ax.imshow(pivot.to_numpy(float), aspect="auto", cmap="YlOrRd")
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_title("Distribution Shift vs Train (PSI-like, lower is closer)")
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            ax.text(j, i, f"{pivot.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
    fig.colorbar(im, ax=ax, shrink=0.85)
    plt.tight_layout()
    plt.savefig(path, dpi=160)
    plt.show()

plot_grouped_bars("mean", SENSOR_COLS, CHART_DIR / "chart_01_sensor_means.png", "Sensor Means by Dataset")
plot_grouped_bars("std", SENSOR_COLS, CHART_DIR / "chart_02_sensor_stds.png", "Sensor Standard Deviations by Dataset")
plot_hist_overlay("A_mag", CHART_DIR / "chart_03_accel_magnitude_overlay.png", "Acceleration Magnitude Distribution")
plot_hist_overlay("G_mag", CHART_DIR / "chart_04_gyro_magnitude_overlay.png", "Gyroscope Magnitude Distribution")
plot_shift_heatmap(CHART_DIR / "chart_05_shift_heatmap_vs_train.png")

In [ ]:
summary = stats_df[stats_df.feature.isin(["Ax", "Ay", "Az", "Gx", "Gy", "Gz", "A_mag", "G_mag"])][
    ["dataset", "feature", "mean", "std", "p05", "median", "p95"]
]

print("Main practical reading:")
print("- synthetic_codex is the closest labeled synthetic dataset to train_all_users.")
print("- production_all has similar total acceleration magnitude to train, but axes are shifted by orientation.")
print("- production_all has larger Gy/Gz spread, meaning more variable rotation behavior.")
print("- synthetic_production is useful for stress/drift testing, but not as labeled train-like synthetic data.")

display(summary)
display(shift_df.sort_values("simple_shift_score", ascending=False).head(20))

## Train vs Production Only

`A_mag` means **accelerometer magnitude**:

`A_mag = sqrt(Ax^2 + Ay^2 + Az^2)`

It is the total acceleration strength across all three accelerometer axes. Its unit is `m/s^2`. When the device is mostly still, it is usually close to gravity, around `9.8 m/s^2`. It is useful because it is less sensitive to phone/watch orientation, but it hides which axis the movement happened on.

`G_mag` means **gyroscope magnitude**:

`G_mag = sqrt(Gx^2 + Gy^2 + Gz^2)`

It is the total rotation speed across all three gyroscope axes. Its unit is `deg/s`. It is useful for seeing how much rotation/motion exists overall, but it hides whether the rotation happened around `Gx`, `Gy`, or `Gz`.

In [ ]:
train_prod_datasets = ["train_all_users", "production_all"]
train_prod_features = SENSOR_COLS + ["A_mag", "G_mag"]

train_prod_stats = stats_df[
    stats_df.dataset.isin(train_prod_datasets) & stats_df.feature.isin(train_prod_features)
][["dataset", "feature", "mean", "std", "p05", "median", "p95"]]

train_prod_shift = shift_df[
    (shift_df.dataset == "production_all") & shift_df.feature.isin(train_prod_features)
][["feature", "mean_delta_in_train_std", "std_ratio", "psi_vs_train", "simple_shift_score"]].sort_values(
    "simple_shift_score", ascending=False
)

print("Train vs production only:")
print("- A_mag is close: total acceleration strength is similar.")
print("- Ax/Ay/Az differ: gravity and orientation are distributed differently across axes.")
print("- Gy/Gz std is larger in production: production has more variable wrist/device rotation.")

display(train_prod_stats)
display(train_prod_shift)

def plot_train_prod_bars(metric, features, path, title):
    view = stats_df[stats_df.dataset.isin(train_prod_datasets) & stats_df.feature.isin(features)]
    pivot = view.pivot(index="feature", columns="dataset", values=metric)
    ax = pivot.plot(kind="bar", figsize=(9, 5), width=0.72)
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel(metric)
    ax.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.savefig(path, dpi=160)
    plt.show()

def plot_train_prod_hist(feature, path, title):
    plt.figure(figsize=(9, 5))
    for dataset in train_prod_datasets:
        values = samples[dataset][feature].dropna()
        lo, hi = np.nanpercentile(values, [1, 99])
        plt.hist(values.clip(lo, hi), bins=80, density=True, histtype="step", linewidth=1.8, label=dataset)
    plt.title(title)
    plt.xlabel(feature)
    plt.ylabel("density")
    plt.grid(alpha=0.2)
    plt.legend()
    plt.tight_layout()
    plt.savefig(path, dpi=160)
    plt.show()

def plot_train_prod_shift(path):
    view = train_prod_shift.sort_values("psi_vs_train", ascending=False)
    ax = view.plot(x="feature", y="psi_vs_train", kind="bar", figsize=(9, 5), legend=False)
    ax.set_title("Production Shift vs Train by Feature")
    ax.set_xlabel("")
    ax.set_ylabel("PSI-like shift, lower is closer")
    ax.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.savefig(path, dpi=160)
    plt.show()

plot_train_prod_bars("mean", SENSOR_COLS, CHART_DIR / "chart_08_train_vs_production_means.png", "Train vs Production: Sensor Means")
plot_train_prod_bars("std", SENSOR_COLS, CHART_DIR / "chart_09_train_vs_production_stds.png", "Train vs Production: Sensor Standard Deviations")
plot_train_prod_hist("A_mag", CHART_DIR / "chart_10_train_vs_production_accel_magnitude.png", "Train vs Production: A_mag Distribution")
plot_train_prod_hist("G_mag", CHART_DIR / "chart_11_train_vs_production_gyro_magnitude.png", "Train vs Production: G_mag Distribution")
plot_train_prod_shift(CHART_DIR / "chart_12_train_vs_production_shift.png")

## Per-class similarity diagnostic

The earlier section compares production against **train_all_users** (the average of 11 activities).
This section splits the train side by class and asks two narrower, descriptive questions:

- For the simple magnitude features `A_mag` and `G_mag`, which train classes does production sit closest to?
- How much of each dataset is "quiet wrist" (low rotation), and how does that compare across classes?

**Important caveat.** `A_mag` and `G_mag` are diagnostic helper features only. They collapse 6 sensor
channels into 2 numbers and ignore orientation, axis-level structure, frequency content, and the
windowed inputs the classifier actually sees. The rankings below describe **distributional
similarity in this reduced space** — they do **not** prove which class the model will predict, and
they should not be used on their own to explain model behavior. For real prediction-collapse
analysis, use the model's own logits / confusion matrix on production windows.

In [ ]:
train_path = ROOT / "data" / "processed" / "fused_all_users_data_labeled_50Hz.csv"
LABEL_COL = "activity"

train_df = pd.read_csv(train_path, usecols=SENSOR_COLS + [LABEL_COL, "User"])
train_df[ACCEL_COLS] = train_df[ACCEL_COLS].apply(pd.to_numeric, errors="coerce")
train_df[GYRO_COLS] = train_df[GYRO_COLS].apply(pd.to_numeric, errors="coerce")
train_df["A_mag"] = np.sqrt(np.square(train_df[ACCEL_COLS]).sum(axis=1))
train_df["G_mag"] = np.sqrt(np.square(train_df[GYRO_COLS]).sum(axis=1))

class_stats = (
    train_df.groupby(LABEL_COL)[FEATURE_COLS]
    .agg(["mean", "std"])
    .round(3)
)
class_stats.columns = [f"{c}_{s}" for c, s in class_stats.columns]
class_stats["row_count"] = train_df.groupby(LABEL_COL).size()
class_stats = class_stats.sort_values("G_mag_mean")
class_stats.to_csv(OUT_DIR / "train_per_class_stats.csv")

print("Train per-class summary (sorted by G_mag mean — quiet wrist at the top):")
class_stats[["A_mag_mean", "A_mag_std", "G_mag_mean", "G_mag_std", "row_count"]]

In [ ]:
# Per-class shift table: how close is production to EACH train class?
# Smallest score => the class whose simple magnitude distribution is closest to production.
PER_CLASS_FEATURES = ["A_mag", "G_mag"]
RNG_PER_CLASS = np.random.default_rng(RNG_SEED + 1)
PER_CLASS_SAMPLE = 40_000

class_samples = {}
for activity, group in train_df.groupby(LABEL_COL):
    group = group[FEATURE_COLS].dropna()
    if len(group) > PER_CLASS_SAMPLE:
        group = group.sample(n=PER_CLASS_SAMPLE, random_state=int(RNG_PER_CLASS.integers(0, 2**31 - 1)))
    class_samples[activity] = group.reset_index(drop=True)

prod_sample = samples["production_all"]

per_class_rows = []
for activity, sample in class_samples.items():
    for feature in PER_CLASS_FEATURES:
        ref = sample[feature].to_numpy(float)
        cur = prod_sample[feature].to_numpy(float)
        ref_mean, ref_std = float(np.mean(ref)), float(np.std(ref))
        cur_mean, cur_std = float(np.mean(cur)), float(np.std(cur))
        per_class_rows.append({
            "train_class": activity,
            "feature": feature,
            "train_mean": ref_mean,
            "prod_mean": cur_mean,
            "mean_delta_in_train_std": abs(cur_mean - ref_mean) / ref_std if ref_std > 1e-9 else np.nan,
            "std_ratio": cur_std / ref_std if ref_std > 1e-9 else np.nan,
            "wasserstein_approx": wasserstein_approx(ref, cur),
            "psi_vs_class": psi(ref, cur),
        })

per_class_df = pd.DataFrame(per_class_rows)
per_class_df["shift_score"] = (
    per_class_df["mean_delta_in_train_std"].fillna(0)
    + np.abs(np.log(per_class_df["std_ratio"].replace(0, np.nan))).fillna(0)
    + per_class_df["psi_vs_class"].fillna(0)
)

ranking = (
    per_class_df.groupby("train_class")["shift_score"].sum()
    .sort_values(ascending=True)
    .rename("total_shift_score_vs_production")
    .to_frame()
)
ranking.to_csv(OUT_DIR / "production_to_train_class_ranking.csv")

print("Train classes ranked by simple magnitude similarity to production (lower = more similar):")
ranking

In [ ]:
# Bar chart of the per-class similarity ranking on (A_mag, G_mag) only.
# Descriptive: lower bar = closer in the magnitude-feature space. Not a model prediction claim.
fig, ax = plt.subplots(figsize=(10, 5.5))
ranking_sorted = ranking.sort_values("total_shift_score_vs_production")
colors = ["#d62728" if i == 0 else "#1f77b4" for i in range(len(ranking_sorted))]
ax.barh(ranking_sorted.index, ranking_sorted["total_shift_score_vs_production"], color=colors)
ax.invert_yaxis()
ax.set_xlabel("total shift score on (A_mag, G_mag) — lower = closer in this reduced feature space")
ax.set_title("Per-class similarity to production on A_mag / G_mag\n(diagnostic only — not a prediction claim)")
ax.grid(axis="x", alpha=0.25)
for i, (cls, val) in enumerate(zip(ranking_sorted.index, ranking_sorted["total_shift_score_vs_production"])):
    ax.text(val, i, f"  {val:.2f}", va="center", fontsize=9)
plt.tight_layout()
plt.savefig(CHART_DIR / "chart_13_production_to_train_class_ranking.png", dpi=160)
plt.show()

In [ ]:
# Nearest-class assignment per production session in the (A_mag mean, A_mag std, G_mag mean, G_mag std) space.
# This is a coarse 4-number centroid match. It does NOT use the model and is not a prediction.
class_centroids = {
    activity: np.array([
        sample["A_mag"].mean(), sample["A_mag"].std(),
        sample["G_mag"].mean(), sample["G_mag"].std(),
    ])
    for activity, sample in class_samples.items()
}
centroid_matrix = np.vstack(list(class_centroids.values()))
centroid_scale = centroid_matrix.std(axis=0) + 1e-9
class_names = list(class_centroids.keys())

prod_files = discover_production_files()
nearest_rows = []
for path in prod_files:
    rename_map = canonical_column_map(path)
    usecols = list(rename_map.keys())
    convert_accel = needs_accel_conversion(path, usecols, rename_map)
    raw = pd.read_csv(path, usecols=usecols)
    frame = clean_chunk(raw, rename_map, convert_accel)
    vec = np.array([
        frame["A_mag"].mean(), frame["A_mag"].std(),
        frame["G_mag"].mean(), frame["G_mag"].std(),
    ])
    distances = np.linalg.norm((centroid_matrix - vec) / centroid_scale, axis=1)
    order = np.argsort(distances)
    nearest_rows.append({
        "session": path.stem.replace("fused_", "").replace("_50Hz", ""),
        "nearest_class": class_names[order[0]],
        "second_class": class_names[order[1]],
        "nearest_distance": float(distances[order[0]]),
        "A_mag_mean": float(vec[0]),
        "G_mag_mean": float(vec[2]),
    })

nearest_df = pd.DataFrame(nearest_rows)
nearest_df.to_csv(OUT_DIR / "production_session_nearest_class.csv", index=False)

class_counts = nearest_df["nearest_class"].value_counts()
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(class_counts.index, class_counts.values, color="#1f77b4")
ax.set_title(f"Nearest train class per production session in (A_mag, G_mag) space — diagnostic only "
             f"({len(nearest_df)} sessions)")
ax.set_ylabel("session count")
ax.grid(axis="y", alpha=0.25)
for x, y in zip(class_counts.index, class_counts.values):
    ax.text(x, y, str(y), ha="center", va="bottom", fontsize=10)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(CHART_DIR / "chart_14_production_nearest_class_per_session.png", dpi=160)
plt.show()

print("Per-session nearest class (centroid match on 4 magnitude features — not a model prediction):")
nearest_df

In [ ]:
# Window-level "quiet wrist" fraction: fraction of 2 s windows where G_mag std < 5 deg/s.
# Descriptive only - characterizes how still each dataset is, not which class the model predicts.
# Important: train windows are built inside each (activity, User) segment so we do not create fake
# windows by stitching different users or non-contiguous sampled rows together.
QUIET_WINDOW_LEN = 100   # 2 seconds at 50 Hz
QUIET_THRESHOLD = 5.0

def quiet_fraction_from_series(g_mag_series, window=QUIET_WINDOW_LEN, threshold=QUIET_THRESHOLD):
    arr = pd.to_numeric(g_mag_series, errors="coerce").dropna().to_numpy(float)
    if len(arr) < window:
        return np.nan, 0
    n_windows = len(arr) // window
    arr = arr[: n_windows * window].reshape(n_windows, window)
    quiet = arr.std(axis=1) < threshold
    return int(quiet.sum()), int(n_windows)

def quiet_fraction_for_parts(parts, window=QUIET_WINDOW_LEN, threshold=QUIET_THRESHOLD):
    quiet_total = 0
    window_total = 0
    for part in parts:
        quiet_count, n_windows = quiet_fraction_from_series(part, window=window, threshold=threshold)
        if n_windows:
            quiet_total += quiet_count
            window_total += n_windows
    return float(quiet_total / window_total) if window_total else np.nan

quiet_rows = []
for activity, group in train_df.groupby(LABEL_COL, sort=False):
    parts = [user_group["G_mag"] for _, user_group in group.groupby("User", sort=False)]
    quiet_rows.append({
        "group": f"train::{activity}",
        "kind": "train_class",
        "quiet_fraction": quiet_fraction_for_parts(parts),
    })

prod_quiet_per_session = []
for path in prod_files:
    rename_map = canonical_column_map(path)
    usecols = list(rename_map.keys())
    convert_accel = needs_accel_conversion(path, usecols, rename_map)
    frame = clean_chunk(pd.read_csv(path, usecols=usecols), rename_map, convert_accel)
    qf = quiet_fraction_for_parts([frame["G_mag"]])
    prod_quiet_per_session.append(qf)

quiet_rows.append({
    "group": "production_all (mean of sessions)",
    "kind": "production",
    "quiet_fraction": float(np.nanmean(prod_quiet_per_session)),
})
quiet_df = pd.DataFrame(quiet_rows).sort_values("quiet_fraction", ascending=False)
quiet_df.to_csv(OUT_DIR / "quiet_wrist_fraction.csv", index=False)

fig, ax = plt.subplots(figsize=(10, 5.5))
colors = ["#d62728" if r.kind == "production" else "#1f77b4" for r in quiet_df.itertuples()]
ax.barh(quiet_df["group"], quiet_df["quiet_fraction"], color=colors)
ax.invert_yaxis()
ax.set_xlim(0, 1)
ax.set_xlabel(f"fraction of 2 s windows with G_mag std < {QUIET_THRESHOLD} deg/s")
ax.set_title("How much of each dataset is 'quiet wrist'? (red = production, blue = train classes)\n"
             "Descriptive only - does not explain model predictions.")
ax.grid(axis="x", alpha=0.25)
for i, v in enumerate(quiet_df["quiet_fraction"]):
    ax.text(v, i, f"  {v:.2f}", va="center", fontsize=9)
plt.tight_layout()
plt.savefig(CHART_DIR / "chart_15_quiet_wrist_fraction.png", dpi=160)
plt.show()

print("Quiet wrist fraction (sorted):")
quiet_df

### How to read the "quiet wrist" chart

Each dataset is split into non-overlapping 2 s windows (100 samples at 50 Hz). For train data,
windows are created **inside each `(activity, User)` segment**, so the notebook does not stitch
unrelated users or activities into fake windows. For each window, we compute the standard deviation
of `G_mag` (gyroscope magnitude). A window is **"quiet"** if its `G_mag` std is below `5 deg/s`.

**What the bars mean**

A larger value means that activity/session has more low-rotation windows. A smaller value means the
wrist rotates more continuously or has frequent spikes. Production is shown as the mean quiet-window
fraction across production sessions.

**Why this is useful**

It shows how much low-motion data exists in production. If production contains many low-motion
windows, an 11-class classifier with no explicit "idle / null" class still has to map those windows
onto one of the known classes. This is a labeling-coverage observation, not a class-similarity proof.

**What this chart is not**

- Not a prediction. The model sees 6 raw channels over a 4 s window, not a single `G_mag` std.
- Not a proof of `hand_tapping` or any other class.
- Not orientation-aware. `G_mag` collapses three gyroscope axes into one number.

In [ ]:
# Box plots — easier to read than 5 overlapping line histograms.
def boxplot_feature(feature, path, title):
    data, labels = [], []
    for dataset, sample in samples.items():
        s = sample[feature].dropna()
        lo, hi = np.nanpercentile(s, [1, 99])
        data.append(s.clip(lo, hi).to_numpy())
        labels.append(dataset)
    fig, ax = plt.subplots(figsize=(10, 5))
    bp = ax.boxplot(data, labels=labels, showfliers=False, patch_artist=True)
    for patch, color in zip(bp["boxes"], ["#1f77b4", "#2ca02c", "#9467bd", "#ff7f0e", "#d62728"]):
        patch.set_facecolor(color)
        patch.set_alpha(0.55)
    ax.set_title(title)
    ax.set_ylabel(feature)
    ax.grid(axis="y", alpha=0.25)
    plt.xticks(rotation=15, ha="right")
    plt.tight_layout()
    plt.savefig(path, dpi=160)
    plt.show()

boxplot_feature("A_mag", CHART_DIR / "chart_16_box_A_mag_by_dataset.png",
                "A_mag distribution by dataset (box = IQR, line = median)")
boxplot_feature("G_mag", CHART_DIR / "chart_17_box_G_mag_by_dataset.png",
                "G_mag distribution by dataset (box = IQR, line = median)")

In [ ]:
# Time-series strip chart: 10 seconds of Ax and Gx for each context, eyeball-comparable.
SNIP_SECONDS = 10
SNIP_LEN = SNIP_SECONDS * 50

def first_n(series, n):
    arr = pd.to_numeric(series, errors="coerce").dropna().to_numpy(float)
    return arr[:n] if len(arr) >= n else arr

snippets = []
for activity in ["sitting", "hand_tapping", "smoking", "hand_scratching"]:
    sub = train_df[train_df[LABEL_COL] == activity]
    snippets.append((f"train::{activity}", first_n(sub["Ax"], SNIP_LEN), first_n(sub["Gx"], SNIP_LEN)))

prod_path = prod_files[0] if prod_files else None
if prod_path is not None:
    rename_map = canonical_column_map(prod_path)
    usecols = list(rename_map.keys())
    convert_accel = needs_accel_conversion(prod_path, usecols, rename_map)
    pf = clean_chunk(pd.read_csv(prod_path, usecols=usecols, nrows=SNIP_LEN * 4), rename_map, convert_accel)
    snippets.append((f"production::{prod_path.stem.replace('fused_', '').replace('_50Hz','')}",
                     first_n(pf["Ax"], SNIP_LEN), first_n(pf["Gx"], SNIP_LEN)))

synth_prod_files = discover_synthetic_production_files()
if synth_prod_files:
    sp = synth_prod_files[0]
    rename_map = canonical_column_map(sp)
    usecols = list(rename_map.keys())
    convert_accel = needs_accel_conversion(sp, usecols, rename_map)
    spf = clean_chunk(pd.read_csv(sp, usecols=usecols, nrows=SNIP_LEN * 4), rename_map, convert_accel)
    snippets.append((f"synthetic_production::{sp.stem}", first_n(spf["Ax"], SNIP_LEN), first_n(spf["Gx"], SNIP_LEN)))

n_rows = len(snippets)
fig, axes = plt.subplots(n_rows, 2, figsize=(13, 1.6 * n_rows + 1), sharex=True)
if n_rows == 1:
    axes = np.array([axes])
t = np.arange(SNIP_LEN) / 50.0
for i, (label, ax_sig, gx_sig) in enumerate(snippets):
    axes[i, 0].plot(t[: len(ax_sig)], ax_sig, color="#1f77b4")
    axes[i, 0].set_ylabel(label, fontsize=8)
    axes[i, 0].grid(alpha=0.2)
    axes[i, 1].plot(t[: len(gx_sig)], gx_sig, color="#d62728")
    axes[i, 1].grid(alpha=0.2)
axes[0, 0].set_title("Ax (m/s^2)")
axes[0, 1].set_title("Gx (deg/s)")
axes[-1, 0].set_xlabel("seconds")
axes[-1, 1].set_xlabel("seconds")
plt.tight_layout()
plt.savefig(CHART_DIR / "chart_18_time_series_strips.png", dpi=160)
plt.show()

In [ ]:
# Gravity sanity + final scorecard. One row per dataset, three numbers, plain English summary.
sanity_rows = []
for name, sample in samples.items():
    a = sample["A_mag"].dropna()
    g = sample["G_mag"].dropna()
    sanity_rows.append({
        "dataset": name,
        "A_mag_median": float(a.median()) if len(a) else np.nan,
        "A_mag_iqr": float(a.quantile(0.75) - a.quantile(0.25)) if len(a) else np.nan,
        "G_mag_median": float(g.median()) if len(g) else np.nan,
    })

shift_score_by_dataset = (
    shift_df.groupby("dataset")["simple_shift_score"].sum().to_dict()
)
sanity_df = pd.DataFrame(sanity_rows)
sanity_df["total_shift_vs_train"] = sanity_df["dataset"].map(shift_score_by_dataset).fillna(0.0)
sanity_df["gravity_ok (~9.8)"] = sanity_df["A_mag_median"].between(8.5, 11.0)
sanity_df.to_csv(OUT_DIR / "scorecard.csv", index=False)

closest = ranking.index[0]
closest_score = float(ranking.iloc[0, 0])
prod_quiet = float(np.nanmean(prod_quiet_per_session))

print("=== Headline (descriptive, on A_mag/G_mag only) ===")
print(f"Closest train class to production in (A_mag, G_mag) space: {closest}  "
      f"(total shift score = {closest_score:.2f})")
print(f"Quiet-wrist fraction for production (mean of sessions): {prod_quiet:.2f}")
print(f"All datasets have gravity in 8.5–11.0 m/s^2 range: {bool(sanity_df['gravity_ok (~9.8)'].all())}")
print()
print("Note: this ranking uses two diagnostic magnitude features only. It describes distribution")
print("similarity in that reduced space — it does not prove which class the model will predict.")
print("To explain model behavior, run the trained model on production windows and inspect logits.")
print()
print("Scorecard:")
sanity_df

## Direct per-axis comparison (no magnitude collapsing)

`A_mag` / `G_mag` are convenient but lossy: two motions with very different axis content can land
on the same magnitude. The next cells compare the **6 raw channels directly** — `Ax, Ay, Az, Gx,
Gy, Gz` — between each train class and production.

Caveat: per-axis comparison only carries activity meaning **if the device orientation is the same
across datasets**. A large `Ax` mean delta can mean "the watch is worn rotated" rather than "the
activity is different". Use these charts together with the magnitude charts: magnitude tells you
if the energy is similar, per-axis tells you if the geometry is similar.

In [ ]:
# Per-axis sample-level shift table: PSI / Wasserstein / mean-delta for each of the 6 raw channels
# between every train class and the production sample.
prod_axis_sample = samples["production_all"]

per_axis_rows = []
for activity, sample in class_samples.items():
    for axis in SENSOR_COLS:
        ref = sample[axis].dropna().to_numpy(float)
        cur = prod_axis_sample[axis].dropna().to_numpy(float)
        ref_mean, ref_std = float(np.mean(ref)), float(np.std(ref))
        cur_mean, cur_std = float(np.mean(cur)), float(np.std(cur))
        per_axis_rows.append({
            "train_class": activity,
            "axis": axis,
            "train_mean": ref_mean,
            "prod_mean": cur_mean,
            "mean_delta_in_train_std": abs(cur_mean - ref_mean) / ref_std if ref_std > 1e-9 else np.nan,
            "std_ratio": cur_std / ref_std if ref_std > 1e-9 else np.nan,
            "wasserstein_approx": wasserstein_approx(ref, cur),
            "psi_vs_class": psi(ref, cur),
        })

per_axis_df = pd.DataFrame(per_axis_rows)
per_axis_df["shift_score"] = (
    per_axis_df["mean_delta_in_train_std"].fillna(0)
    + np.abs(np.log(per_axis_df["std_ratio"].replace(0, np.nan))).fillna(0)
    + per_axis_df["psi_vs_class"].fillna(0)
)
per_axis_df.to_csv(OUT_DIR / "per_axis_class_vs_production_shift.csv", index=False)

# Heatmap: rows = train class, cols = 6 axes, value = PSI vs production. Lower = closer.
heatmap = per_axis_df.pivot(index="train_class", columns="axis", values="psi_vs_class")[SENSOR_COLS]
fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(heatmap.to_numpy(float), aspect="auto", cmap="YlOrRd")
ax.set_xticks(range(len(SENSOR_COLS)))
ax.set_xticklabels(SENSOR_COLS)
ax.set_yticks(range(len(heatmap.index)))
ax.set_yticklabels(heatmap.index)
ax.set_title("Per-axis distribution shift: each train class vs production (PSI, lower = closer)")
for i in range(len(heatmap.index)):
    for j in range(len(SENSOR_COLS)):
        ax.text(j, i, f"{heatmap.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(im, ax=ax, shrink=0.85, label="PSI")
plt.tight_layout()
plt.savefig(CHART_DIR / "chart_19_per_axis_class_vs_production_psi.png", dpi=160)
plt.show()

# Per-class total per-axis shift, ranked.
per_axis_total = (
    per_axis_df.groupby("train_class")["shift_score"].sum()
    .sort_values()
    .rename("total_per_axis_shift_vs_production")
    .to_frame()
)
per_axis_total.to_csv(OUT_DIR / "per_axis_class_ranking.csv")

print("Train classes ranked by per-axis distance to production (sums over Ax,Ay,Az,Gx,Gy,Gz):")
per_axis_total

In [ ]:
# Window-level per-axis features: for each 4 s window, compute mean and std on each of the 6 channels.
# This matches the Stage 3 / Stage 4 model window length: 200 samples at 50 Hz.
# Important: train windows are built inside each (activity, User) segment, not from random samples.
MODEL_WINDOW_LEN = 200  # 4 s @ 50 Hz

def windowed_axis_features(frame, window_len=MODEL_WINDOW_LEN):
    arr = frame[SENSOR_COLS].apply(pd.to_numeric, errors="coerce").to_numpy(float)
    n = (len(arr) // window_len) * window_len
    if n == 0:
        return pd.DataFrame(columns=[f"{c}_{s}" for c in SENSOR_COLS for s in ("mean", "std")])
    arr = arr[:n].reshape(-1, window_len, len(SENSOR_COLS))
    means = np.nanmean(arr, axis=1)
    stds = np.nanstd(arr, axis=1)
    data = {}
    for j, ch in enumerate(SENSOR_COLS):
        data[f"{ch}_mean"] = means[:, j]
        data[f"{ch}_std"] = stds[:, j]
    return pd.DataFrame(data)

# Build train window-feature frames per activity, preserving user/activity segment boundaries.
window_feats = {}
for activity, group in train_df.groupby(LABEL_COL, sort=False):
    parts = []
    for _, user_group in group.groupby("User", sort=False):
        wf = windowed_axis_features(user_group[SENSOR_COLS])
        if not wf.empty:
            parts.append(wf)
    window_feats[f"train::{activity}"] = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()

# Build production windows per production session.
prod_window_parts = []
for path in prod_files:
    rename_map = canonical_column_map(path)
    usecols = list(rename_map.keys())
    convert_accel = needs_accel_conversion(path, usecols, rename_map)
    frame = clean_chunk(pd.read_csv(path, usecols=usecols), rename_map, convert_accel)
    wf = windowed_axis_features(frame[SENSOR_COLS])
    if not wf.empty:
        prod_window_parts.append(wf)
window_feats["production_all"] = pd.concat(prod_window_parts, ignore_index=True)

feature_cols = [f"{c}_{s}" for c in SENSOR_COLS for s in ("mean", "std")]

# Window-level per-feature PSI: train class windows vs production windows.
ref_window = window_feats["production_all"]
window_rows = []
for group_name, df_g in window_feats.items():
    if group_name == "production_all" or df_g.empty:
        continue
    for feat in feature_cols:
        a = df_g[feat].dropna().to_numpy(float)
        b = ref_window[feat].dropna().to_numpy(float)
        window_rows.append({
            "train_class": group_name.replace("train::", ""),
            "feature": feat,
            "psi_vs_production": psi(a, b),
            "wasserstein_vs_production": wasserstein_approx(a, b),
        })

window_df = pd.DataFrame(window_rows)
window_df.to_csv(OUT_DIR / "windowed_per_axis_features_vs_production.csv", index=False)

# Heatmap: rows = train class, cols = 12 window features, value = PSI vs production windows.
hm = window_df.pivot(index="train_class", columns="feature", values="psi_vs_production")[feature_cols]
fig, ax = plt.subplots(figsize=(13, 6))
im = ax.imshow(hm.to_numpy(float), aspect="auto", cmap="YlOrRd")
ax.set_xticks(range(len(feature_cols)))
ax.set_xticklabels(feature_cols, rotation=45, ha="right")
ax.set_yticks(range(len(hm.index)))
ax.set_yticklabels(hm.index)
ax.set_title("4 s window-level per-axis features: each train class vs production (PSI, lower = closer)")
for i in range(len(hm.index)):
    for j in range(len(feature_cols)):
        ax.text(j, i, f"{hm.iloc[i, j]:.1f}", ha="center", va="center", fontsize=7)
fig.colorbar(im, ax=ax, shrink=0.85, label="PSI")
plt.tight_layout()
plt.savefig(CHART_DIR / "chart_20_windowed_per_axis_psi.png", dpi=160)
plt.show()

window_ranking = (
    window_df.groupby("train_class")["psi_vs_production"].sum()
    .sort_values()
    .rename("total_window_feature_psi_vs_production")
    .to_frame()
)
window_ranking.to_csv(OUT_DIR / "windowed_per_axis_class_ranking.csv")

print("Train classes ranked by 4 s window-level per-axis distance to production (sum of PSI over 12 features):")
window_ranking

### How to read the per-axis charts

**chart_19 — sample-level per-axis PSI heatmap.** Rows are the 11 train classes, columns are the
six raw channels (`Ax, Ay, Az, Gx, Gy, Gz`), and each cell is the PSI between that class's
distribution on that channel and production's distribution on the same channel. Smaller values
(lighter cells) mean the two distributions overlap more on that single channel. A row that is
uniformly light is a class whose raw channel distributions match production well; a row with one
hot column highlights a single axis where the class differs strongly from production.

**chart_20 — window-level per-axis features.** For each 4 s model-length window we compute the per-window
mean and std on each of the 6 channels (12 features total), then PSI those window features
against production windows. This is much closer to what the classifier actually sees than the
sample-level heatmap, because the model never sees individual samples in isolation — it sees
windows.

**Interpretation rules of thumb.**

- A class that is close on **`*_mean`** features matches production in static posture / orientation.
- A class that is close on **`*_std`** features matches production in motion energy.
- A class can be close on `_std` and far on `_mean` — that means similar movement intensity but
  different wrist orientation. Re-running with gravity-aligned axes would be the next step.
- The two new rankings (`per_axis_class_ranking.csv` and `windowed_per_axis_class_ranking.csv`)
  may disagree with the magnitude-based ranking. That disagreement is the point: each metric
  measures a different aspect of similarity, and the model uses none of them directly. Use these
  as triangulation, not as a substitute for running the trained model on production windows.

## Feature-by-feature side-by-side: training vs production

Each individual feature carries more discriminative signal than `A_mag` / `G_mag`. The next cells
build a single side-by-side table - **training column on the left, production column on the right**
- with mean and std for every raw sensor channel and every windowed per-axis feature, plus the
delta and the production-to-train std ratio. The same comparison is then plotted as paired bars so
it can be read at a glance.


In [ ]:
# Side-by-side feature table: training (left) vs production (right).
# Covers raw 6 channels and all 12 windowed per-axis features (mean and std on each axis).
train_window_feats = windowed_axis_features(train_df[SENSOR_COLS])
prod_window_feats = window_feats["production_all"]

side_rows = []

for axis in SENSOR_COLS:
    t = pd.to_numeric(train_df[axis], errors="coerce").dropna()
    p = pd.to_numeric(samples["production_all"][axis], errors="coerce").dropna()
    side_rows.append({
        "feature": axis,
        "scope": "sample",
        "train_mean": float(t.mean()),
        "train_std": float(t.std()),
        "prod_mean": float(p.mean()),
        "prod_std": float(p.std()),
    })

for col in [f"{c}_{s}" for c in SENSOR_COLS for s in ("mean", "std")]:
    t = train_window_feats[col].dropna()
    p = prod_window_feats[col].dropna()
    side_rows.append({
        "feature": col,
        "scope": "window",
        "train_mean": float(t.mean()),
        "train_std": float(t.std()),
        "prod_mean": float(p.mean()),
        "prod_std": float(p.std()),
    })

side_df = pd.DataFrame(side_rows)
side_df["mean_delta"] = side_df["prod_mean"] - side_df["train_mean"]
side_df["std_ratio_prod_over_train"] = side_df["prod_std"] / side_df["train_std"].replace(0, np.nan)
side_df["abs_mean_delta_in_train_std"] = (
    side_df["mean_delta"].abs() / side_df["train_std"].replace(0, np.nan)
)

side_df = side_df[[
    "scope", "feature",
    "train_mean", "train_std",
    "prod_mean", "prod_std",
    "mean_delta", "abs_mean_delta_in_train_std", "std_ratio_prod_over_train",
]].round(4)
side_df.to_csv(OUT_DIR / "feature_side_by_side_train_vs_production.csv", index=False)

print("Side-by-side feature comparison (training | production):")
side_df


In [ ]:
# Paired bar chart: training (blue, left) vs production (red, right) for every feature.
# One row of subplots per scope (sample / window) so the y-scales stay readable.
def plot_paired(features_df, title, save_path):
    n = len(features_df)
    fig, axes = plt.subplots(2, 1, figsize=(max(10, n * 0.45), 8.5), sharex=False)

    means = features_df[["feature", "train_mean", "prod_mean"]].set_index("feature")
    stds = features_df[["feature", "train_std", "prod_std"]].set_index("feature")

    x = np.arange(len(means))
    w = 0.4

    axes[0].bar(x - w/2, means["train_mean"], width=w, label="train", color="#1f77b4")
    axes[0].bar(x + w/2, means["prod_mean"], width=w, label="production", color="#d62728")
    axes[0].axhline(0, color="black", linewidth=0.7)
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(means.index, rotation=45, ha="right", fontsize=8)
    axes[0].set_ylabel("mean")
    axes[0].set_title(f"{title} - feature mean (train left, production right)")
    axes[0].legend()
    axes[0].grid(axis="y", alpha=0.25)

    axes[1].bar(x - w/2, stds["train_std"], width=w, label="train", color="#1f77b4")
    axes[1].bar(x + w/2, stds["prod_std"], width=w, label="production", color="#d62728")
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(stds.index, rotation=45, ha="right", fontsize=8)
    axes[1].set_ylabel("std")
    axes[1].set_title(f"{title} - feature std (train left, production right)")
    axes[1].legend()
    axes[1].grid(axis="y", alpha=0.25)

    plt.tight_layout()
    plt.savefig(save_path, dpi=160)
    plt.show()

sample_view = side_df[side_df["scope"] == "sample"].reset_index(drop=True)
window_view = side_df[side_df["scope"] == "window"].reset_index(drop=True)

plot_paired(sample_view, "Raw 6 channels", CHART_DIR / "chart_21_paired_raw_channels.png")
plot_paired(window_view, "Windowed per-axis features (4 s)", CHART_DIR / "chart_22_paired_window_features.png")


### How to read the side-by-side table and paired bars

- **Same color, different bar height** => the feature distribution shifted between train and production.
- **`abs_mean_delta_in_train_std` > ~0.5** => the production mean is far enough from the training mean
  that the windowed features will look like a different region of the input space. The model has not
  seen many windows there, so it has to extrapolate.
- **`std_ratio_prod_over_train` >> 1** => production has more variance on that feature than training.
  Production sees motions / orientations the training distribution rarely produced.
- **`std_ratio_prod_over_train` << 1** => production is more concentrated than training - usually
  caused by lots of low-motion windows.
- Comparing **`Ax`/`Ay`/`Az` mean** between sides shows orientation drift directly. If `Az` flips sign
  or shrinks toward zero, the watch is being worn at a different tilt than during training.
- Comparing **`*_std` window features** shows motion-energy drift independent of orientation.

This table and these two charts together replace `A_mag`/`G_mag` as the primary diagnostic when the
question is "per-feature, how does production differ from training?". They do not replace running
the trained model on production windows, which is still the only diagnostic that can answer the
prediction-collapse question.


## Gravity and unit-conversion sanity check

Local paper/config check:

- `internal/papers/triage/ICTH_16_extracted.txt` says Garmin accelerometer data is converted from `milli-g` to `m/s^2`.
- The same paragraph says data is downsampled to 50 Hz and windowed into 200 samples with 50% overlap.
- It does not document gravity removal, so gravity removal should stay disabled unless we run a separate A/B experiment.

This cell checks whether each dataset behaves like:

- already in `m/s^2`,
- raw `milliG` that needs `* 9.80665 / 1000`, or
- something else that is not gravity-consistent.

A mostly still wrist should have `A_mag` near gravity, about `9.80665 m/s^2`. This is a unit sanity check, not a classifier explanation.


In [ ]:
# Unit-conversion hypothesis test using accelerometer magnitude and gravity.
# Output files:
# - unit_conversion_gravity_check.csv
# - unit_conversion_gravity_check_by_file.csv
# - charts/chart_21_unit_conversion_gravity_check.png

GRAVITY_MPS2 = 9.80665
GRAVITY_OK_LOW = 8.5
GRAVITY_OK_HIGH = 11.0

def raw_a_mag_values(path):
    rename_map = canonical_column_map(path)
    usecols = list(rename_map.keys())
    values = []
    for chunk in pd.read_csv(path, usecols=usecols, chunksize=CHUNK_SIZE):
        frame = chunk.rename(columns=rename_map)[ACCEL_COLS].apply(pd.to_numeric, errors="coerce")
        values.append(np.sqrt(np.square(frame).sum(axis=1)).to_numpy(float))
    return np.concatenate(values) if values else np.array([], dtype=float)

def selected_unit_assumption(raw_median, millig_median):
    if GRAVITY_OK_LOW <= raw_median <= GRAVITY_OK_HIGH:
        return "already_mps2", raw_median, "already_mps2_gravity_ok"
    if raw_median > 80 and GRAVITY_OK_LOW <= millig_median <= GRAVITY_OK_HIGH:
        return "milliG_to_mps2", millig_median, "raw_milliG_conversion_needed_and_ok"
    if raw_median > 80:
        return "milliG_to_mps2", millig_median, "raw_milliG_but_converted_gravity_not_ok"
    return "as_is", raw_median, "as_is_closest_but_gravity_not_ok"

gravity_rows = []
gravity_file_rows = []
for dataset, files in datasets.items():
    all_raw = []
    for path in files:
        raw_a = raw_a_mag_values(path)
        all_raw.append(raw_a)
        gravity_file_rows.append({
            "dataset": dataset,
            "file": str(path.relative_to(ROOT)),
            "rows": int(len(raw_a)),
            "raw_A_mag_median": float(np.nanmedian(raw_a)) if len(raw_a) else np.nan,
            "if_milliG_to_mps2_A_mag_median": float(np.nanmedian(raw_a * MILLI_G_TO_MPS2)) if len(raw_a) else np.nan,
        })

    raw = np.concatenate(all_raw) if all_raw else np.array([], dtype=float)
    raw_median = float(np.nanmedian(raw)) if len(raw) else np.nan
    millig_median = float(np.nanmedian(raw * MILLI_G_TO_MPS2)) if len(raw) else np.nan
    g_median = float(np.nanmedian(raw * GRAVITY_MPS2)) if len(raw) else np.nan
    assumption, selected_median, status = selected_unit_assumption(raw_median, millig_median)

    gravity_rows.append({
        "dataset": dataset,
        "files": len(files),
        "rows": int(len(raw)),
        "raw_A_mag_median": raw_median,
        "if_milliG_to_mps2_A_mag_median": millig_median,
        "if_g_to_mps2_A_mag_median": g_median,
        "selected_unit_assumption": assumption,
        "selected_A_mag_median_mps2": selected_median,
        "distance_from_gravity_mps2": abs(selected_median - GRAVITY_MPS2),
        "gravity_ok_after_selected_assumption": bool(GRAVITY_OK_LOW <= selected_median <= GRAVITY_OK_HIGH),
        "unit_gravity_status": status,
    })

gravity_check_df = pd.DataFrame(gravity_rows)
gravity_check_df.to_csv(OUT_DIR / "unit_conversion_gravity_check.csv", index=False)
pd.DataFrame(gravity_file_rows).to_csv(OUT_DIR / "unit_conversion_gravity_check_by_file.csv", index=False)

fig, ax = plt.subplots(figsize=(10, 4.8))
colors = ["#2ca02c" if ok else "#d62728" for ok in gravity_check_df["gravity_ok_after_selected_assumption"]]
ax.bar(gravity_check_df["dataset"], gravity_check_df["selected_A_mag_median_mps2"], color=colors, alpha=0.82)
ax.axhline(GRAVITY_MPS2, color="#111111", linestyle="--", linewidth=1.2, label="gravity 9.80665 m/s^2")
ax.axhspan(GRAVITY_OK_LOW, GRAVITY_OK_HIGH, color="#2ca02c", alpha=0.08, label="simple ok band 8.5-11.0")
ax.set_ylabel("selected A_mag median (m/s^2)")
ax.set_title("Gravity/unit conversion sanity check")
ax.grid(axis="y", alpha=0.25)
ax.legend(loc="upper right", fontsize=8)
plt.xticks(rotation=15, ha="right")
for i, row in enumerate(gravity_check_df.itertuples(index=False)):
    ax.text(i, row.selected_A_mag_median_mps2, f"{row.selected_A_mag_median_mps2:.2f}\n{row.selected_unit_assumption}",
            ha="center", va="bottom", fontsize=8)
plt.tight_layout()
plt.savefig(CHART_DIR / "chart_21_unit_conversion_gravity_check.png", dpi=160)
plt.show()

print("Gravity/unit conversion check:")
print("- green = selected unit assumption lands near gravity")
print("- red = even the best common unit assumption is not gravity-consistent")
gravity_check_df[[
    "dataset",
    "raw_A_mag_median",
    "if_milliG_to_mps2_A_mag_median",
    "selected_unit_assumption",
    "selected_A_mag_median_mps2",
    "gravity_ok_after_selected_assumption",
    "unit_gravity_status",
]]